นำเข้า Library และตั้งค่าตัวแปรพื้นฐาน

In [ ]:
import pyrealsense2 as rs
import numpy as np
import matplotlib.pyplot as plt
import cv2

# ตั้งค่าพารามิเตอร์ของ 2D Occupancy Grid Map
RESOLUTION = 0.05  # ความละเอียด 5 ซม. ต่อ 1 ช่องตาราง
MAP_SIZE_Z = 3.0   # มองไปข้างหน้า 3 เมตร
MAP_SIZE_X = 3.0   # มองกว้างซ้าย-ขวา รวม 3 เมตร (-1.5 ถึง 1.5 เมตร)

GRID_W = int(MAP_SIZE_X / RESOLUTION)
GRID_H = int(MAP_SIZE_Z / RESOLUTION)

# ช่วงความสูงที่จะชน (Y-axis) (หน่วย: เมตร)
# สมมติกล้องตั้งอยู่ระดับพื้น 0
MIN_HEIGHT = -0.2 
MAX_HEIGHT = 0.5  

print(f"ขนาด Grid Map: กว้าง {GRID_W} ช่อง x สูง {GRID_H} ช่อง")

เตรียม Pipeline และดึง Intrinsics

In [ ]:
# ชื่อไฟล์ .bag ที่ต้องการอ่าน (ตรวจสอบให้แน่ใจว่าชื่อตรงกัน)
BAG_FILE = "dataset_10sec.bag" 

pipeline = rs.pipeline()
config = rs.config()
rs.config.enable_device_from_file(config, BAG_FILE, repeat_playback=False)

profile = pipeline.start(config)

# ดึงค่า Intrinsics และ Scale
depth_stream = profile.get_stream(rs.stream.depth)
intrinsics = depth_stream.as_video_stream_profile().get_intrinsics()
depth_scale = profile.get_device().first_depth_sensor().get_depth_scale()

# เตรียมฟิลเตอร์ทำความสะอาด
spatial = rs.spatial_filter()
temporal = rs.temporal_filter()
hole_filling = rs.hole_filling_filter()

print("เตรียม Pipeline และดึง Intrinsics สำเร็จ")

ดึงข้อมูล 1 เฟรมมาทำความสะอาด

In [ ]:
# รอให้เฟรมพร้อมและข้ามไปสัก 10 เฟรม (จำลอง Warm-up)
for _ in range(10):
    frames = pipeline.wait_for_frames()

# ดึงเฟรมปัจจุบัน
depth_frame = frames.get_depth_frame()

# ผ่านฟิลเตอร์ (ลำดับมีความสำคัญ: Spatial -> Temporal -> Hole Filling)
depth_frame = spatial.process(depth_frame)
depth_frame = temporal.process(depth_frame)
depth_frame = hole_filling.process(depth_frame)

# แปลงเป็น Numpy Array และคูณสเกลเพื่อให้หน่วยเป็น 'เมตร'
depth_image = np.asanyarray(depth_frame.get_data())
z_meters = depth_image * depth_scale

print(f"ดึงข้อมูลสำเร็จ: ขนาด {z_meters.shape}, ค่าความลึกสูงสุด {np.max(z_meters):.2f} เมตร")

# ปิด Pipeline หลังจากดึงรูปเสร็จแล้ว
pipeline.stop()